In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Subtract, Multiply, concatenate, Dot
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

DATASET_SLUG  = "siamese-data"   # <-- GANTI sesuai nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'final_' if USE_AUGMENTED else ''
meta_f = 'final_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Hyperparameter (tuned via Optuna, best MAE=1.8615 on 4-fold proxy) ────────
BILSTM_UNITS  = 128
DROPOUT       = 0.40
EPOCHS        = 150
BATCH_SIZE    = 16
PATIENCE      = 15
LR            = 2.68e-3
N_SEEDS       = 3

# ── Ordinal Regression Loss & Metric ─────────────────────────────────────────
def ordinal_loss(y_true, y_pred):
    """Sum of binary cross-entropies: apakah grade > k? (k=1..9)"""
    thresholds   = tf.cast(tf.range(1, 10), tf.float32)
    y_true_exp   = tf.expand_dims(tf.cast(y_true, tf.float32), -1)
    y_binary     = tf.cast(y_true_exp > thresholds, tf.float32)
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_binary, y_pred)
    )

def ordinal_mae(y_true, y_pred):
    """Grade prediction = jumlah threshold yang terlampaui + 1."""
    grade_pred = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=-1) + 1.0
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - grade_pred))


def attention_pool(seq_out, name_prefix):
    """Soft attention pooling atas output BiLSTM (return_sequences=True)."""
    score   = Dense(1, activation='tanh', use_bias=False,
                    name=f'{name_prefix}_attn_score')(seq_out)
    weights = Lambda(lambda x: tf.nn.softmax(x, axis=1),
                    name=f'{name_prefix}_attn_w')(score)
    pooled  = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=1),
                    name=f'{name_prefix}_attn_pool')([seq_out, weights])
    return pooled


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                n_scalar=10,
                bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Siamese BiLSTM v11 — Ordinal Regression + Normalized + Centroid
                         + cos_sim_q_a + orisinalitas

    Perbedaan dari v10:
    - Tambahkan kembali cos_sim_q_a dan orisinalitas (= 1 - cos_sim_q_a)
      yang sebelumnya hilang saat migrasi ke arsitektur direct.
    - cos_sim_q_a  : cosine similarity antara repr. question dan answer (BiLSTM)
                     → mengukur relevansi jawaban terhadap pertanyaan
    - orisinalitas : 1 - cos_sim_q_a
                     → penalti jika jawaban hanya menyalin ulang pertanyaan
    - Sesuai tujuan penelitian: model mengevaluasi kemiripan jawaban terhadap
      kunci jawaban SEKALIGUS relevansinya terhadap pertanyaan.

    Merged: [ea(256), eak(256), eq(256), abs_diff(256), had_prod(256),
             cos_sim_ak_a(1), cos_sim_q_a(1), orisinalitas(1),
             scalar_dense(32)]  = 1315D
    Head: Dense(512) → Dropout → Dense(64) → Dense(9, sigmoid)  [ordinal]
    """
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=True), name='bilstm_shared'
    )

    inp_q      = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak     = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a      = Input(shape=(a_seq_len,  emb_dim), name='inp_a')
    inp_scalar = Input(shape=(n_scalar,),            name='inp_scalar')

    # ── BiLSTM + Attention Pooling ────────────────────────────────────────────
    eq_seq  = shared_bilstm(inp_q)
    eak_seq = shared_bilstm(inp_ak)
    ea_seq  = shared_bilstm(inp_a)

    eq  = attention_pool(eq_seq,  'q')
    eak = attention_pool(eak_seq, 'ak')
    ea  = attention_pool(ea_seq,  'a')

    # ── Kemiripan answerkey vs answer (task utama Siamese) ───────────────────
    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    # ── Relevansi jawaban terhadap pertanyaan (penalti menyalin soal) ─────────
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_sim_q_a')([eq, ea])
    orisinalitas = Lambda(lambda x: 1.0 - x, name='orisinalitas')(cos_sim_q_a)

    # ── Scalar branch: normalized + centroid ─────────────────────────────────
    scalar_feat = Dense(32, activation='relu', name='scalar_dense')(inp_scalar)

    merged = concatenate(
        [ea, eak, eq, abs_diff, had_prod,
         cos_sim_ak_a, cos_sim_q_a, orisinalitas,
         scalar_feat],
        name='merged'
    )

    x   = Dense(512, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64, activation='relu')(x)
    out = Dense(9, activation='sigmoid', name='ordinal_out')(x)

    model = Model(
        inputs=[inp_q, inp_ak, inp_a, inp_scalar],
        outputs=out,
        name='siamese_bilstm_v11'
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=ordinal_loss,
        metrics=[ordinal_mae]
    )
    return model


# Verifikasi arsitektur (n_scalar=10: 5 normalized + 5 centroid)
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2],
    n_scalar   = 10
)
_tmp.summary()
del _tmp

In [ ]:
# ── Helper functions + Precompute Scalar Features ─────────────────────────────
from sklearn.metrics import cohen_kappa_score

def compute_scalar_features_all(answers_emb, answerkeys_emb, verbose=True):
    n = answers_emb.shape[0]
    feats = np.zeros((n, 5), dtype=np.float32)
    if verbose:
        print(f"Precomputing scalar features untuk {n} sampel...")
    for i in range(n):
        if verbose and i % 500 == 0:
            print(f"  {i}/{n}")
        a  = answers_emb[i].astype(np.float64)
        ak = answerkeys_emb[i].astype(np.float64)

        ak_norm = ak / (np.linalg.norm(ak, axis=-1, keepdims=True) + 1e-8)
        a_norm  = a  / (np.linalg.norm(a,  axis=-1, keepdims=True) + 1e-8)

        ak_mask = np.abs(ak).sum(axis=-1) > 1e-6
        a_mask  = np.abs(a ).sum(axis=-1) > 1e-6
        ak_n    = ak_norm[ak_mask]
        a_n     = a_norm[a_mask]

        if ak_n.shape[0] == 0 or a_n.shape[0] == 0:
            continue

        sim = ak_n @ a_n.T
        rec = float(sim.max(axis=1).mean())
        pre = float(sim.max(axis=0).mean())
        f1  = 2.0 * rec * pre / (rec + pre + 1e-8)

        m_ak = ak_n.mean(axis=0)
        m_a  = a_n.mean(axis=0)
        cos  = float(m_ak @ m_a / (np.linalg.norm(m_ak) * np.linalg.norm(m_a) + 1e-8))
        lrat = float(a_mask.sum()) / max(float(ak_mask.sum()), 1.0)

        feats[i] = [rec, pre, f1, cos, lrat]

    if verbose:
        print(f"  Selesai. Shape: {feats.shape}")
    return feats


def make_scalar_input_train(feats, idpsj_ids):
    norm_part = np.zeros_like(feats)
    cent_part = np.zeros_like(feats)
    for idpsj in np.unique(idpsj_ids):
        m = idpsj_ids == idpsj
        mu = feats[m].mean(axis=0)
        sg = feats[m].std(axis=0) + 1e-8
        norm_part[m] = (feats[m] - mu) / sg
        cent_part[m] = mu
    norm_part = np.clip(norm_part, -3.0, 3.0)
    return np.hstack([norm_part, cent_part]).astype(np.float32)


def make_scalar_input_test(feats):
    mu = feats.mean(axis=0)
    sg = feats.std(axis=0) + 1e-8
    norm_part = np.clip((feats - mu) / sg, -3.0, 3.0)
    cent_part = np.tile(mu, (len(feats), 1))
    return np.hstack([norm_part, cent_part]).astype(np.float32)


def ordinal_predict(model, X_q, X_ak, X_a, X_scalar=None):
    inputs = [X_q, X_ak, X_a]
    if X_scalar is not None:
        inputs.append(X_scalar)
    sigmoid_out = model.predict(inputs, verbose=0)
    grade = np.sum(sigmoid_out > 0.5, axis=-1) + 1
    return grade.astype(np.float32), sigmoid_out


# ── Precompute scalar features (1x untuk semua sampel) ───────────────────────
all_scalar_feats = compute_scalar_features_all(answers_emb, answerkeys_emb)
N_SCALAR = all_scalar_feats.shape[1] * 2   # 5 normalized + 5 centroid = 10

# ── Setup variabel LOPO ───────────────────────────────────────────────────────
idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_all      = metadata['grade'].values.astype(np.float32)

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

print(f"N_SCALAR={N_SCALAR} | {len(idpsj_list)} IDPSJ | {is_real.sum()} sampel real")

In [ ]:
# ── LOPO Cross-Validation + Ensemble ─────────────────────────────────────────
fold_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}")

    train_idx = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx   = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  Val: {len(y_val)}  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    train_idpsj_ids = metadata['IDPSJ'].values[train_idx]
    scalar_tr  = make_scalar_input_train(all_scalar_feats[train_idx], train_idpsj_ids)
    scalar_val = make_scalar_input_test(all_scalar_feats[val_idx])
    scalar_te  = make_scalar_input_test(all_scalar_feats[test_idx])

    grade_int          = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map           = dict(zip(unique_g, counts_g))
    n_kelas            = len(unique_g)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w           = raw_w / raw_w.mean()

    # ── Ensemble: N_SEEDS run per fold ────────────────────────────────────────
    all_sigmoid_raw = []

    for seed in range(N_SEEDS):
        print(f"\n  -- Seed {seed+1}/{N_SEEDS} --")
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model = build_model(
            q_seq_len    = questions_emb.shape[1],
            ak_seq_len   = answerkeys_emb.shape[1],
            a_seq_len    = answers_emb.shape[1],
            emb_dim      = answers_emb.shape[2],
            n_scalar     = N_SCALAR,
            bilstm_units = BILSTM_UNITS,
            dropout      = DROPOUT
        )

        reduce_lr  = ReduceLROnPlateau(monitor='val_ordinal_mae', factor=0.5,
                                       patience=3, min_lr=1e-6, mode='min', verbose=0)
        early_stop = EarlyStopping(monitor='val_ordinal_mae', patience=PATIENCE,
                                   restore_best_weights=True, mode='min', verbose=1)

        model.fit(
            [X_q_tr, X_ak_tr, X_a_tr, scalar_tr], y_train,
            sample_weight=sample_w,
            validation_data=([X_q_val, X_ak_val, X_a_val, scalar_val], y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[reduce_lr, early_stop], verbose=1
        )

        grade_s, sigmoid_s = ordinal_predict(model, X_q_te, X_ak_te, X_a_te, scalar_te)
        all_sigmoid_raw.append(sigmoid_s)

        mae_s = mean_absolute_error(y_test, np.clip(grade_s, 1, 10))
        print(f"  Seed {seed+1} MAE: {mae_s:.4f}")

    # ── Rata-rata sigmoid ensemble → grade final ──────────────────────────────
    mean_sigmoid = np.mean(all_sigmoid_raw, axis=0)
    y_pred_final = np.clip(
        np.round(np.sum(mean_sigmoid > 0.5, axis=-1) + 1), 1, 10
    ).astype(np.float32)

    mae_final  = mean_absolute_error(y_test, y_pred_final)
    rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
    qwk_final  = cohen_kappa_score(y_test.astype(int), y_pred_final.astype(int),
                                   weights='quadratic')

    print(f"\n  MAE: {mae_final:.4f}  |  RMSE: {rmse_final:.4f}  |  QWK: {qwk_final:.4f}")

    fold_results.append({
        'fold'      : i + 1,
        'test_idpsj': test_id,
        'val_idpsj' : val_id,
        'n_train'   : len(y_train),
        'n_val'     : len(y_val),
        'n_test'    : len(y_test),
        'mae'       : mae_final,
        'rmse'      : rmse_final,
        'qwk'       : qwk_final,
        'y_test'    : y_test,
        'y_pred'    : y_pred_final,
    })

    model_path = os.path.join(OUT_DIR, f'model_fold_{i+1:02d}.keras')
    model.save(model_path)
    print(f"  Model saved -> {model_path}")

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")

In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

summary = pd.DataFrame([{
    'fold'      : r['fold'],
    'test_idpsj': r['test_idpsj'],
    'n_test'    : r['n_test'],
    'MAE'       : round(r['mae'],  4),
    'RMSE'      : round(r['rmse'], 4),
    'QWK'       : round(r['qwk'],  4),
} for r in fold_results])

print("=" * 65)
print("Hasil per Fold — Siamese BiLSTM v11 (Ordinal + Norm + Centroid)")
print("=" * 65)
print(summary.to_string(index=False))
print(f"\nMAE  : {summary['MAE'].mean():.4f}  ±  {summary['MAE'].std():.4f}")
print(f"RMSE : {summary['RMSE'].mean():.4f}  ±  {summary['RMSE'].std():.4f}")
print(f"QWK  : {summary['QWK'].mean():.4f}  ±  {summary['QWK'].std():.4f}")

summary.to_csv(os.path.join(OUT_DIR, 'lopo_results_v11.csv'), index=False)

y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])
x          = np.arange(len(summary))

# ── Gambar 1: MAE per Fold ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['MAE'], color='steelblue', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['MAE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean MAE = {summary['MAE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('MAE')
ax.set_title('MAE per Fold — Siamese BiLSTM v11')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_plot1_mae_per_fold.png'), dpi=150)
plt.show()

# ── Gambar 2: Scatter Plot Prediksi vs Aktual ────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_all_true, y_all_pred, alpha=0.4, edgecolors='k', linewidths=0.3)
ax.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax.set_xlabel('Grade Aktual')
ax.set_ylabel('Grade Prediksi')
ax.set_title(f'Prediksi vs Aktual — Siamese BiLSTM v11\n'
             f'MAE={summary["MAE"].mean():.4f}  '
             f'RMSE={summary["RMSE"].mean():.4f}  '
             f'QWK={summary["QWK"].mean():.4f}')
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_plot2_scatter.png'), dpi=150)
plt.show()

# ── Gambar 3: RMSE per Fold ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['RMSE'], color='mediumseagreen', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['RMSE'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Mean RMSE = {summary['RMSE'].mean():.4f}")
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('RMSE')
ax.set_title('RMSE per Fold — Siamese BiLSTM v11')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_plot3_rmse_per_fold.png'), dpi=150)
plt.show()

# ── Gambar 4: QWK per Fold ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, summary['QWK'], color='mediumpurple', alpha=0.85,
       edgecolor='black', linewidth=0.5)
ax.axhline(summary['QWK'].mean(), color='darkviolet', linestyle='--', linewidth=1.5,
           label=f"Mean QWK = {summary['QWK'].mean():.4f}")
ax.axhline(0.6, color='gray', linestyle=':', linewidth=1.2, label='Threshold 0.6 (substantial)')
ax.set_xticks(x)
ax.set_xticklabels(summary['test_idpsj'].astype(str))
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('QWK')
ax.set_ylim(0, 1)
ax.set_title('Quadratic Weighted Kappa per Fold — Siamese BiLSTM v11')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'v11_plot4_qwk_per_fold.png'), dpi=150)
plt.show()

print("\nGambar disimpan:")
for name in ['v11_plot1_mae_per_fold.png', 'v11_plot2_scatter.png',
             'v11_plot3_rmse_per_fold.png', 'v11_plot4_qwk_per_fold.png']:
    print(f"  {os.path.join(OUT_DIR, name)}")


In [ ]:
# ── Selective Fold Re-run untuk Analisis Thesis (Bagian 4.5.2) ────────────────
# Jalankan cell ini TERPISAH dari cell LOPO di atas.
# Prasyarat: jalankan semua cell sampai "Helper functions + Precompute Scalar Features".
#
# Fold 16 (IDPSJ 16): MAE=1.1964, QWK=0.8088  → Fold TERBAIK  (MAE terendah)
# Fold 11 (IDPSJ 11): MAE=1.2667, QWK=0.8127  → Fold TERBAIK  (QWK tertinggi)
# Fold 6  (IDPSJ  6): MAE=2.3944, QWK=0.3490  → Fold TERBURUK (QWK terendah ke-2)
# Fold 9  (IDPSJ  9): MAE=2.7184, QWK=0.3096  → Fold TERBURUK (QWK terendah)

TARGET_IDPSJ = [6, 9, 11, 16]

FOLD_LABELS = {
    6 : "TERBURUK ke-2 (QWK=0.3490, MAE=2.3944)",
    9 : "TERBURUK     (QWK=0.3096, MAE=2.7184)",
    11: "TERBAIK      (QWK=0.8127, MAE=1.2667)",
    16: "TERBAIK ke-2 (QWK=0.8088, MAE=1.1964)",
}

# ── Cari datafinal_preprocessed.csv di semua dataset yang ter-mount ──────────
import glob as _glob

_candidates = _glob.glob('/kaggle/input/**/datafinal_preprocessed.csv', recursive=True)
if not _candidates:
    raise FileNotFoundError(
        "datafinal_preprocessed.csv tidak ditemukan di /kaggle/input/.\n"
        "Pastikan dataset yang berisi file ini sudah ditambahkan ke notebook Kaggle."
    )
TEXT_CSV_PATH = _candidates[0]
print(f"Menggunakan CSV: {TEXT_CSV_PATH}")

# ── Load teks asli dari CSV ───────────────────────────────────────────────────
df_text = pd.read_csv(TEXT_CSV_PATH)
df_text['IDJwb'] = df_text['IDJwb'].astype(str)
df_text = df_text.set_index('IDJwb')[['questions_clean', 'answerKeys_clean', 'answer_clean']]

selective_results = {}

for i, test_id in enumerate(idpsj_list):
    if test_id not in TARGET_IDPSJ:
        continue

    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test IDPSJ={test_id}  |  Val IDPSJ={val_id}")

    train_idx = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx   = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  Val: {len(y_val)}  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    train_idpsj_ids = metadata['IDPSJ'].values[train_idx]
    scalar_tr  = make_scalar_input_train(all_scalar_feats[train_idx], train_idpsj_ids)
    scalar_val = make_scalar_input_test(all_scalar_feats[val_idx])
    scalar_te  = make_scalar_input_test(all_scalar_feats[test_idx])

    grade_int          = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map           = dict(zip(unique_g, counts_g))
    n_kelas            = len(unique_g)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w           = raw_w / raw_w.mean()

    all_sigmoid_raw = []
    for seed in range(N_SEEDS):
        print(f"\n  -- Seed {seed+1}/{N_SEEDS} --")
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model = build_model(
            q_seq_len    = questions_emb.shape[1],
            ak_seq_len   = answerkeys_emb.shape[1],
            a_seq_len    = answers_emb.shape[1],
            emb_dim      = answers_emb.shape[2],
            n_scalar     = N_SCALAR,
            bilstm_units = BILSTM_UNITS,
            dropout      = DROPOUT
        )

        reduce_lr  = ReduceLROnPlateau(monitor='val_ordinal_mae', factor=0.5,
                                       patience=3, min_lr=1e-6, mode='min', verbose=0)
        early_stop = EarlyStopping(monitor='val_ordinal_mae', patience=PATIENCE,
                                   restore_best_weights=True, mode='min', verbose=1)

        model.fit(
            [X_q_tr, X_ak_tr, X_a_tr, scalar_tr], y_train,
            sample_weight=sample_w,
            validation_data=([X_q_val, X_ak_val, X_a_val, scalar_val], y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[reduce_lr, early_stop], verbose=1
        )

        _, sigmoid_s = ordinal_predict(model, X_q_te, X_ak_te, X_a_te, scalar_te)
        all_sigmoid_raw.append(sigmoid_s)
        tf.keras.backend.clear_session()

    mean_sigmoid = np.mean(all_sigmoid_raw, axis=0)
    y_pred_final = np.clip(
        np.round(np.sum(mean_sigmoid > 0.5, axis=-1) + 1), 1, 10
    ).astype(np.float32)

    mae_s  = mean_absolute_error(y_test, y_pred_final)
    rmse_s = np.sqrt(mean_squared_error(y_test, y_pred_final))
    qwk_s  = cohen_kappa_score(y_test.astype(int), y_pred_final.astype(int), weights='quadratic')
    print(f"\n  → MAE={mae_s:.4f}  RMSE={rmse_s:.4f}  QWK={qwk_s:.4f}")

    # ── Gabungkan prediksi dengan teks asli ───────────────────────────────────
    test_meta = metadata.iloc[test_idx][['IDJwb', 'IDPSJ', 'grade']].copy()
    test_meta['IDJwb']      = test_meta['IDJwb'].astype(str)
    test_meta['grade_pred'] = y_pred_final.astype(int)
    test_meta['error']      = np.abs(y_test - y_pred_final).astype(int)
    test_meta = test_meta.join(df_text, on='IDJwb', how='left')
    test_meta = test_meta.sort_values('error').reset_index(drop=True)

    out_path = os.path.join(OUT_DIR, f'thesis_predictions_idpsj{test_id}.csv')
    test_meta.to_csv(out_path, index=False)
    print(f"  Saved -> {out_path}")

    selective_results[test_id] = test_meta

# ── Tampilan ringkasan untuk Thesis ──────────────────────────────────────────
cols = ['IDJwb', 'grade', 'grade_pred', 'error', 'answer_clean']
for idpsj_id in sorted(selective_results.keys()):
    df_res = selective_results[idpsj_id]
    print(f"\n{'='*70}")
    print(f"IDPSJ {idpsj_id} — {FOLD_LABELS[idpsj_id]}")
    print(f"\n5 Prediksi Paling Akurat (error terkecil):")
    print(df_res[cols].head(5).to_string(index=False, max_colwidth=80))
    print(f"\n5 Prediksi Paling Meleset (error terbesar):")
    print(df_res[cols].tail(5).to_string(index=False, max_colwidth=80))